In [5]:
# Getting my dataset ready

## Starting with the network variables
from coinmetrics.api_client import CoinMetricsClient
import pandas as pd
import os

START_DATE = "2011-01-01"
END_DATE = "2026-01-01"

# API
client = CoinMetricsClient(api_key=None)

# Daily frequency just in case
df = (
    client.get_asset_metrics(
        assets="btc",
        metrics=["AdrActCnt", "TxCnt", "TxTfrCnt"],
        start_time=START_DATE,
        end_time=END_DATE,
        frequency="1d"
    )
    .to_dataframe()
)

df["time"] = pd.to_datetime(df["time"], utc=True)
for col in ["AdrActCnt", "TxCnt", "TxTfrCnt"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.rename(
    columns={
        "asset": "coin",
        "AdrActCnt": "active_addresses",
        "TxCnt": "transaction_count",
        "TxTfrCnt": "payment_transfer_count",
    }
)

df = df.drop(columns=["coin"])

print(df.head())
print(df.tail())

os.makedirs("Dataset", exist_ok=True)
df.to_csv("Dataset/Dataset.csv", index=False)

                       time  active_addresses  transaction_count  \
0 2011-01-01 00:00:00+00:00              1071                473   
1 2011-01-02 00:00:00+00:00              1097                418   
2 2011-01-03 00:00:00+00:00              1405                997   
3 2011-01-04 00:00:00+00:00              1111                842   
4 2011-01-05 00:00:00+00:00              1497               1161   

   payment_transfer_count  
0                     842  
1                     716  
2                    1456  
3                    1289  
4                    1711  
                          time  active_addresses  transaction_count  \
5475 2025-12-28 00:00:00+00:00            572908             540696   
5476 2025-12-29 00:00:00+00:00            693511             548253   
5477 2025-12-30 00:00:00+00:00            749500             366458   
5478 2025-12-31 00:00:00+00:00            697063             407824   
5479 2026-01-01 00:00:00+00:00            570904             469075 

In [6]:
# Now I import S&P 500 closing price data from Yahoo Finance, find daily returns and add it to the main dataset

import pandas as pd
import yfinance as yf

df = pd.read_csv("Dataset/Dataset.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df["date"] = df["time"].dt.date

## Downloading Yahoo Finance Data and making sure it's in the correct shape
START_DATE = "2011-01-01"
END_DATE = "2026-01-02"

sp500 = yf.download(
    "^GSPC",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

sp500 = sp500.reset_index()

# I will only keep the relevant columns
sp500["stock_market_return"] = sp500["Close"].pct_change()
sp500["date"] = pd.to_datetime(sp500["Date"]).dt.date
sp500 = sp500[["date", "stock_market_return"]]

# Merge with the main dataset
df = df.merge(sp500, on="date", how="left")
df = df.drop(columns=["date"])

df.to_csv("Dataset/Dataset.csv", index=False)

In [7]:
import pandas as pd
import os

# THe directories
aux_path = "Dataset"                       # the subfolder just for my dataset
core_file = os.path.join(aux_path, "Dataset.csv")
output_file = os.path.join(aux_path, "Dataset.csv")

trend_file_map = {
    "Bitcoin_search.csv": "bitcoin_search",
    "Bitcoin_hack_search.csv": "bitcoin_hack_search",
    "Ether_search.csv": "ethereum_search",
    "Ripple_search.csv": "ripple_search",
    "Litecoin_search.csv": "litecoin_search",
    "Tether_search.csv": "tether_search",
    "Bitcoin_Cash_search.csv": "bitcoin_cash_search",
    "Tezos_search.csv": "tezos_search",
    "Binance_Coin_search.csv": "binance_coin_search",
    "Monero_search.csv": "monero_search",
    "Cardano_search.csv": "cardano_search",
}

# Load and match date and time
df = pd.read_csv(core_file)
df["time"] = pd.to_datetime(df["time"], utc=True)
df["year_month"] = df["time"].dt.to_period("M").astype(str)

# Merging with the rf
rf_file = os.path.join(aux_path, "Risk_free.csv")
rf = pd.read_csv(rf_file, skiprows=4)

rf = rf.rename(columns={rf.columns[0]: "year_month_raw"})
rf["year_month_raw"] = rf["year_month_raw"].astype(str).str.strip()
rf = rf[rf["year_month_raw"].str.fullmatch(r"\d{6}")].copy()

rf["year_month"] = pd.to_datetime(
    rf["year_month_raw"], format="%Y%m"
).dt.to_period("M").astype(str)

rf["RF"] = pd.to_numeric(rf["RF"], errors="coerce")
rf["RF_decimal"] = rf["RF"] / 100

rf = rf[["year_month", "RF", "RF_decimal"]]
df = df.merge(rf, on="year_month", how="left")

# Google trends
def load_trend_file(filepath, output_colname):
    trend = pd.read_csv(filepath)

    trend = trend.rename(columns={trend.columns[0]: "Time"})
    trend["Time"] = pd.to_datetime(trend["Time"], errors="coerce")
    trend = trend[trend["Time"].notna()].copy()

    value_col = trend.columns[1]
    trend[output_colname] = pd.to_numeric(trend[value_col], errors="coerce")
    trend["year_month"] = trend["Time"].dt.to_period("M").astype(str)

    trend = trend[["year_month", output_colname]].drop_duplicates(subset=["year_month"])
    return trend

# Merging all of the Google Trends
for filename, colname in trend_file_map.items():
    filepath = os.path.join(aux_path, filename)

    try:
        trend_df = load_trend_file(filepath, colname)
        df = df.merge(trend_df, on="year_month", how="left")
        print(f"Added: {colname}")
    except FileNotFoundError:
        print(f"Missing file: {filepath}")
    except Exception as e:
        print(f"Problem with {filepath}: {e}")

# Save
df = df.drop(columns=["year_month"])
df.to_csv(output_file, index=False)

print("\nDone.")
print(f"Saved to: {output_file}")
print(df.columns.tolist())
print(df.head())

Added: bitcoin_search
Added: bitcoin_hack_search
Added: ethereum_search
Added: ripple_search
Added: litecoin_search
Added: tether_search
Added: bitcoin_cash_search
Added: tezos_search
Added: binance_coin_search
Added: monero_search
Added: cardano_search

Done.
Saved to: Dataset/Dataset.csv
['time', 'active_addresses', 'transaction_count', 'payment_transfer_count', 'stock_market_return', 'RF', 'RF_decimal', 'bitcoin_search', 'bitcoin_hack_search', 'ethereum_search', 'ripple_search', 'litecoin_search', 'tether_search', 'bitcoin_cash_search', 'tezos_search', 'binance_coin_search', 'monero_search', 'cardano_search']
                       time  active_addresses  transaction_count  \
0 2011-01-01 00:00:00+00:00              1071                473   
1 2011-01-02 00:00:00+00:00              1097                418   
2 2011-01-03 00:00:00+00:00              1405                997   
3 2011-01-04 00:00:00+00:00              1111                842   
4 2011-01-05 00:00:00+00:00             

/var/folders/02/hw2xskd17854kzhgq2xw_1hm0000gn/T/ipykernel_1104/2520077427.py:30: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df["time"].dt.to_period("M").astype(str)


In [8]:
# Finally, I add the historical cryptocurrency returns, and formulate the coin market index!
import pandas as pd
import numpy as np

## Historical Data
hist = pd.read_csv("Dataset/historical.csv")

# Cleaning the date and dropping redundant columns
hist["date"] = pd.to_datetime(hist["date"], errors="coerce", utc=True)

hist = hist[["date", "coin_id", "market_cap", "price", "volume_24h", "cmc_rank"]].copy()

hist = hist.dropna(subset=["date", "coin_id", "market_cap", "price", "volume_24h"])

hist = hist.sort_values(["coin_id", "date"]).reset_index(drop=True)

# I only include coins with market caps larger than 1 million USD
hist = hist[hist["market_cap"] > 1_000_000].copy()

# Daily returns
hist["coin_return"] = hist.groupby("coin_id")["price"].pct_change()

# I lag the market cap
hist["lag_market_cap"] = hist.groupby("coin_id")["market_cap"].shift(1)

hist = hist.dropna(subset=["coin_return", "lag_market_cap"])

# Daily coin-market returns
hist["weighted_return_component"] = hist["coin_return"] * hist["lag_market_cap"]

cmkt_daily = (
    hist.groupby("date", as_index=False)
    .agg(
        total_lag_mcap=("lag_market_cap", "sum"),
        total_market_cap=("market_cap", "sum"),
        total_volume_24h=("volume_24h", "sum"),
        n_coins=("coin_id", "nunique"),
        weighted_return_sum=("weighted_return_component", "sum")
    )
)

cmkt_daily["cmkt_return"] = (
    cmkt_daily["weighted_return_sum"] / cmkt_daily["total_lag_mcap"]
)

cmkt_daily = cmkt_daily[
    ["date", "cmkt_return", "total_market_cap", "total_volume_24h", "n_coins"]
].copy()

btc = hist[hist["coin_id"].astype(str).str.lower() == "bitcoin"].copy()

if btc.empty:
    btc = hist[hist["coin_id"].astype(str).str.lower() == "btc"].copy()

btc = btc.sort_values("date").copy()
btc["btc_return"] = btc["price"].pct_change()

btc_daily = btc[["date", "btc_return", "market_cap", "volume_24h"]].copy()
btc_daily = btc_daily.rename(columns={
    "market_cap": "btc_market_cap",
    "volume_24h": "btc_volume_24h"
})

cmkt_daily = cmkt_daily.merge(btc_daily, on="date", how="outer")
cmkt_daily = cmkt_daily.sort_values("date").reset_index(drop=True)

splice_end = pd.Timestamp("2013-04-29", tz="UTC")

pre_mask = cmkt_daily["date"] < splice_end

cmkt_daily.loc[pre_mask, "cmkt_return"] = cmkt_daily.loc[pre_mask, "btc_return"]
cmkt_daily.loc[pre_mask, "total_market_cap"] = cmkt_daily.loc[pre_mask, "btc_market_cap"]
cmkt_daily.loc[pre_mask, "total_volume_24h"] = cmkt_daily.loc[pre_mask, "btc_volume_24h"]
cmkt_daily.loc[pre_mask, "n_coins"] = 1

cmkt_daily = cmkt_daily[
    ["date", "cmkt_return", "total_market_cap", "total_volume_24h", "n_coins"]
].copy()

cmkt_daily = cmkt_daily.dropna(subset=["date", "cmkt_return"])
cmkt_daily = cmkt_daily.sort_values("date").reset_index(drop=True)

cmkt_daily.to_csv("Dataset/cmkt_daily.csv", index=False)

print(cmkt_daily.head())
print(cmkt_daily.tail())
print(cmkt_daily.columns.tolist())

                       date  cmkt_return  total_market_cap  total_volume_24h  \
0 2013-04-29 00:00:00+00:00     0.074196      1.698609e+09               0.0   
1 2013-04-30 00:00:00+00:00    -0.037084      1.636299e+09               0.0   
2 2013-05-01 00:00:00+00:00    -0.156481      1.380825e+09               0.0   
3 2013-05-02 00:00:00+00:00    -0.101493      1.240230e+09               0.0   
4 2013-05-03 00:00:00+00:00    -0.072274      1.151009e+09               0.0   

   n_coins  
0        7  
1        7  
2        7  
3        6  
4        6  
                          date  cmkt_return  total_market_cap  \
3010 2021-07-27 00:00:00+00:00     0.040348      1.607681e+12   
3011 2021-07-28 00:00:00+00:00     0.009205      1.622596e+12   
3012 2021-07-29 00:00:00+00:00     0.011454      1.641499e+12   
3013 2021-07-30 00:00:00+00:00     0.039271      1.706679e+12   
3014 2021-07-31 00:00:00+00:00     0.003853      1.713599e+12   

      total_volume_24h  n_coins  
3010      1.9166

In [9]:
import pandas as pd

df = pd.read_csv("Dataset/Dataset.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df["date"] = df["time"].dt.normalize()

cmkt = pd.read_csv("Dataset/cmkt_daily.csv")
cmkt["date"] = pd.to_datetime(cmkt["date"], utc=True)

cmkt = cmkt.rename(columns={"cmkt_return": "CMKT"})

df = df.drop(columns=[
    "CMKT", "cmkt_return", "total_market_cap", "total_volume_24h", "n_coins"
], errors="ignore")

# Merge
df = df.merge(cmkt, on="date", how="left")

df = df.drop(columns=["date"])

df.to_csv("Dataset/Dataset.csv", index=False)

print(df.head())
print(df.columns.tolist())

                       time  active_addresses  transaction_count  \
0 2011-01-01 00:00:00+00:00              1071                473   
1 2011-01-02 00:00:00+00:00              1097                418   
2 2011-01-03 00:00:00+00:00              1405                997   
3 2011-01-04 00:00:00+00:00              1111                842   
4 2011-01-05 00:00:00+00:00              1497               1161   

   payment_transfer_count  stock_market_return    RF  RF_decimal  \
0                     842                  NaN  0.01      0.0001   
1                     716                  NaN  0.01      0.0001   
2                    1456                  NaN  0.01      0.0001   
3                    1289            -0.001313  0.01      0.0001   
4                    1711             0.005007  0.01      0.0001   

   bitcoin_search  bitcoin_hack_search  ethereum_search  ...  tether_search  \
0               0                    0                0  ...             36   
1               0       

In [14]:
# Splicing pre-2013 returns with the Coin Market

import pandas as pd
import numpy as np

DATASET_PATH = "Dataset/Dataset.csv"
BTC_PATH = "Dataset/Bitcoin Historical Results Price Data.csv"

df = pd.read_csv(DATASET_PATH)
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.sort_values("time").reset_index(drop=True)
df["date"] = df["time"].dt.normalize()

if "CMKT_original" not in df.columns:
    df["CMKT_original"] = df["CMKT"]

# Daily Bitcoin price data
btc = pd.read_csv(BTC_PATH)

btc["date"] = pd.to_datetime(
    btc["Date"], format="%d/%m/%Y", utc=True, errors="coerce"
).dt.normalize()

btc["Price"] = (
    btc["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)
btc["btc_price_usd"] = pd.to_numeric(btc["Price"], errors="coerce")

btc = (
    btc[["date", "btc_price_usd"]]
    .dropna(subset=["date", "btc_price_usd"])
    .sort_values("date")
    .drop_duplicates("date")
    .reset_index(drop=True)
)

# Computing returns
btc["btc_return"] = btc["btc_price_usd"].pct_change()

for col in ["btc_price_usd", "btc_return", "CMKT_source"]:
    if col in df.columns:
        df = df.drop(columns=col)

df = df.merge(
    btc[["date", "btc_price_usd", "btc_return"]],
    on="date",
    how="left"
)

first_cmkt_date = df.loc[df["CMKT_original"].notna(), "date"].min()

if pd.isna(first_cmkt_date):
    raise ValueError("No non-missing original CMKT found in Dataset/Dataset.csv.")

print("First original CMKT date:", first_cmkt_date)

splice_mask = (
    (df["date"] < first_cmkt_date) &
    (df["CMKT_original"].isna()) &
    (df["btc_return"].notna())
)

df["CMKT"] = df["CMKT_original"]
df.loc[splice_mask, "CMKT"] = df.loc[splice_mask, "btc_return"]

df["CMKT_source"] = None
df.loc[df["CMKT_original"].notna(), "CMKT_source"] = "Original CMKT"
df.loc[splice_mask, "CMKT_source"] = "BTC splice (2011-2013 pre-CMKT)"

df = df.drop(columns=["date"])
df.to_csv(DATASET_PATH, index=False)

check = pd.read_csv(DATASET_PATH)
check["time"] = pd.to_datetime(check["time"], utc=True, errors="coerce")

n_spliced = (check["CMKT_source"] == "BTC splice (2011-2013 pre-CMKT)").sum()
first_spliced = check.loc[check["CMKT_source"] == "BTC splice (2011-2013 pre-CMKT)", "time"].min()
last_spliced = check.loc[check["CMKT_source"] == "BTC splice (2011-2013 pre-CMKT)", "time"].max()
first_final_cmkt = check.loc[check["CMKT"].notna(), "time"].min()

print("\nDone.")
print("Rows filled by BTC splice:", n_spliced)
print("First spliced date:", first_spliced)
print("Last spliced date:", last_spliced)
print("First non-missing final CMKT date:", first_final_cmkt)

print("\nPreview of spliced rows:")
print(
    check.loc[
        check["CMKT_source"] == "BTC splice (2011-2013 pre-CMKT)",
        ["time", "btc_price_usd", "btc_return", "CMKT", "CMKT_source"]
    ].head(10)
)

print("\nBoundary check around CMKT start:")
boundary = check.loc[
    (check["time"] >= pd.Timestamp(first_cmkt_date) - pd.Timedelta(days=3)) &
    (check["time"] <= pd.Timestamp(first_cmkt_date) + pd.Timedelta(days=3)),
    ["time", "btc_return", "CMKT_original", "CMKT", "CMKT_source"]
]
print(boundary)

First original CMKT date: 2013-04-29 00:00:00+00:00

Done.
Rows filled by BTC splice: 848
First spliced date: 2011-01-02 00:00:00+00:00
Last spliced date: 2013-04-28 00:00:00+00:00
First non-missing final CMKT date: 2011-01-02 00:00:00+00:00

Preview of spliced rows:
                        time  btc_price_usd  btc_return  CMKT  \
1  2011-01-02 00:00:00+00:00            0.3         0.0   0.0   
2  2011-01-03 00:00:00+00:00            0.3         0.0   0.0   
3  2011-01-04 00:00:00+00:00            0.3         0.0   0.0   
4  2011-01-05 00:00:00+00:00            0.3         0.0   0.0   
5  2011-01-06 00:00:00+00:00            0.3         0.0   0.0   
6  2011-01-07 00:00:00+00:00            0.3         0.0   0.0   
7  2011-01-08 00:00:00+00:00            0.3         0.0   0.0   
8  2011-01-09 00:00:00+00:00            0.3         0.0   0.0   
9  2011-01-10 00:00:00+00:00            0.3         0.0   0.0   
10 2011-01-11 00:00:00+00:00            0.3         0.0   0.0   

                

In [17]:
# I also need to make sure we have Ripple and Ethereum Returns as well


import pandas as pd
import numpy as np

DATASET_PATH = "Dataset/Dataset.csv"
HISTORICAL_PATH = "Dataset/Historical.csv"

df = pd.read_csv(DATASET_PATH)
hist = pd.read_csv(HISTORICAL_PATH)

df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df["date"] = df["time"].dt.normalize()


hist = hist.copy()

hist["date"] = pd.to_datetime(hist["date"], utc=True, errors="coerce").dt.normalize()
hist["coin_id"] = hist["coin_id"].astype(str).str.strip()
hist["price"] = pd.to_numeric(hist["price"], errors="coerce")

hist = hist.dropna(subset=["date", "coin_id", "price"]).copy()

id_map = {
    "1": "bitcoin",
    "1027": "ethereum",
    "52": "ripple"
}

hist = hist[hist["coin_id"].isin(id_map.keys())].copy()

if hist.empty:
    raise ValueError(
        "No rows found for coin_id 1, 52, or 1027 in Historical.csv."
    )

# Daily returns by coin
hist["coin_name"] = hist["coin_id"].map(id_map)

hist = hist.sort_values(["coin_name", "date"]).copy()

hist["daily_return"] = (
    hist.groupby("coin_name")["price"]
        .pct_change()
)

price_wide = hist.pivot_table(
    index="date",
    columns="coin_name",
    values="price",
    aggfunc="last"
)

ret_wide = hist.pivot_table(
    index="date",
    columns="coin_name",
    values="daily_return",
    aggfunc="last"
)

price_wide = price_wide.rename(columns={
    "bitcoin": "bitcoin_price_usd",
    "ethereum": "ethereum_price_usd",
    "ripple": "ripple_price_usd"
})

ret_wide = ret_wide.rename(columns={
    "bitcoin": "bitcoin_return",
    "ethereum": "ethereum_return",
    "ripple": "ripple_return"
})

coin_panel = price_wide.join(ret_wide, how="outer").reset_index()


cols_to_drop = [
    "bitcoin_price_usd", "ethereum_price_usd", "ripple_price_usd",
    "bitcoin_return", "ethereum_return", "ripple_return"
]

for c in cols_to_drop:
    if c in df.columns:
        df = df.drop(columns=c)

df = df.merge(coin_panel, on="date", how="left")

# I save back to the main dataset
df = df.drop(columns=["date"])
df.to_csv(DATASET_PATH, index=False)

# Making sure everything works
check = pd.read_csv(DATASET_PATH)
check["time"] = pd.to_datetime(check["time"], utc=True, errors="coerce")

print("Non-missing counts:")
for c in [
    "CMKT",
    "bitcoin_return", "ethereum_return", "ripple_return",
    "bitcoin_price_usd", "ethereum_price_usd", "ripple_price_usd"
]:
    if c in check.columns:
        print(f"{c}: {check[c].notna().sum()}")

print("\nDate coverage:")
for c in ["bitcoin_return", "ethereum_return", "ripple_return"]:
    if c in check.columns:
        tmp = check.loc[check[c].notna(), "time"]
        if len(tmp) > 0:
            print(f"{c}: {tmp.min()} --> {tmp.max()}")

print("\nPreview:")
preview_cols = [
    "time", "CMKT",
    "bitcoin_price_usd", "bitcoin_return",
    "ethereum_price_usd", "ethereum_return",
    "ripple_price_usd", "ripple_return"
]
preview_cols = [c for c in preview_cols if c in check.columns]
print(check[preview_cols].head(10))

Non-missing counts:
CMKT: 3863
bitcoin_return: 3015
ethereum_return: 2184
ripple_return: 2917
bitcoin_price_usd: 3016
ethereum_price_usd: 2185
ripple_price_usd: 2918

Date coverage:
bitcoin_return: 2013-04-29 00:00:00+00:00 --> 2021-07-31 00:00:00+00:00
ethereum_return: 2015-08-08 00:00:00+00:00 --> 2021-07-31 00:00:00+00:00
ripple_return: 2013-08-05 00:00:00+00:00 --> 2021-07-31 00:00:00+00:00

Preview:
                       time  CMKT  bitcoin_price_usd  bitcoin_return  \
0 2011-01-01 00:00:00+00:00   NaN                NaN             NaN   
1 2011-01-02 00:00:00+00:00   0.0                NaN             NaN   
2 2011-01-03 00:00:00+00:00   0.0                NaN             NaN   
3 2011-01-04 00:00:00+00:00   0.0                NaN             NaN   
4 2011-01-05 00:00:00+00:00   0.0                NaN             NaN   
5 2011-01-06 00:00:00+00:00   0.0                NaN             NaN   
6 2011-01-07 00:00:00+00:00   0.0                NaN             NaN   
7 2011-01-08 00: